In [ ]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)


# 02 编码器模块 (core.encoders)

基于 `hscredit.xlsx` 真实信贷数据，演示各种编码方式。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (init_setting, WOEEncoder, TargetEncoder, CountEncoder,
    OrdinalEncoder, QuantileEncoder)
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D')
target = 'FPD15'
features = [c for c in df.columns if c not in ['MOB1','MOB2','loan_date','FPD15','SFPD15']]
num_feats = df[features].select_dtypes('number').columns.tolist()

X = df[num_feats].fillna(-999); y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(f'训练集: {X_tr.shape}, 坏率: {y_tr.mean():.2%}')

## 1. WOE 编码

In [ ]:
top5 = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','overdue_loan_m1_count_12m']
woe = WOEEncoder(max_n_bins=8, method='mdlp')
woe.fit(X_tr[top5], y_tr)
X_woe = woe.transform(X_te[top5])
print('WOE编码结果:')
print(X_woe.head(3).round(4))

print('\nWOE映射表（bj_qy24）:')
woe.woe_tables_['bj_qy24'][['分箱标签','好样本数','坏样本数','坏样本率','分档WOE值','分档IV值']]

## 2. Target 编码

In [ ]:
te = TargetEncoder(smoothing=10.0)
te.fit(X_tr[top5], y_tr)
X_te_enc = te.transform(X_te[top5])
print('Target编码结果:')
print(X_te_enc.head(3).round(4))

## 3. Count 编码

In [ ]:
ce = CountEncoder()
ce.fit(X_tr[top5], y_tr)
print('Count编码:')
print(ce.transform(X_te[top5]).head(3))

## 4. Quantile 编码

In [ ]:
qe = QuantileEncoder(n_quantiles=10)
qe.fit(X_tr[top5], y_tr)
print('Quantile编码:')
print(qe.transform(X_te[top5]).head(3))

## 5. Pipeline 中使用 WOE 编码器

In [ ]:
from hscredit import Pipeline
from sklearn.linear_model import LogisticRegression
from hscredit.core.metrics import ks, auc

pipe = Pipeline([
    ('woe', WOEEncoder(max_n_bins=8)),
    ('lr', LogisticRegression(C=0.1, max_iter=1000)),
])
pipe.fit(X_tr, y_tr)
prob = pipe.predict_proba(X_te)[:,1]
print(f'Pipeline WOE+LR  KS:{ks(y_te,prob):.4f}  AUC:{auc(y_te,prob):.4f}')

## 6. 各编码方式 IV 对比

In [ ]:
from hscredit.core.metrics import iv as iv_metric
rows = []
for enc_name, enc in [('WOE', woe), ('Target', te), ('Count', ce), ('Quantile', qe)]:
    try:
        X_enc = enc.transform(X_te[top5])
        for feat in top5:
            iv_val = iv_metric(y_te.values, X_enc[feat].values)
            rows.append({'编码': enc_name, '特征': feat, 'IV': round(iv_val, 4)})
    except Exception as e:
        pass
pd.DataFrame(rows).pivot(index='特征', columns='编码', values='IV').round(4)